> ### Note on Labs and Assigments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# **IS 4487 LAB 7: DATA TRANSFORMATION**

## Outline

- Load and preview the cleaned Megatelco dataset  
- Engineer new columns from existing data  
- Binning to simplify numeric variable values  
- On-hot-Encoding and integer encoding to change categorical into numeric values
- Use log scaling, normalization and standardization
- Try your own transformation logic  

This lab continues from **Lab 6**, where we cleaned the Megatelco dataset.  

Now, we will create new, more useful features for modeling and exploration.

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Labs/lab_07_data_transformation.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

If you're new to Colab: [Colab FAQ](https://research.google.com/colaboratory/faq.html)




## Megatelco Data Dictionary

 DEMOGRAPHIC VARIABLES:
 - College - has the customer attended some college (one, zero)
 - Income - annual income of customer
 - House - estimated price of the customer's home (if applicable)

 USAGE VARIABLES:
 - Data Overage Mb - Average number of megabytes that the customer used in excess of the plan limit (over last 12 months)
 - Data Leftover Mb - Average number of megabytes that the customer use was below the plan limit (over last 12 months)
 - Data Mb Used - Average number of megabytes used per month (over last 12 months)
 - Text Message Count - Average number of texts per month (over last 12 months)
 - Over 15 Minute Calls Per Month - Average number of calls over 15 minutes in duration per month (over last 12 months)
 - Average Call Duration- Average call duration (over last 12 months)

PHONE VARIABLES:
 - Operating System - Current operating system of phone
 - Handset Price - Retail price of the phone used by the customer

ATTITUDINAL VARIABLES:
 - Reported Satisfaction - Survey response to "How satisfied are you with your current phone plan?" (high, avg, low)
 - Reported Usage Level - Survey response to "How much do your use your phone?" (high, avg, low)
 - Considering Change of Plan - Survey response to "Are you currently planning to change companies when your contract expires?" (no, yes)

OTHER VARIABLES
 - Leave - Did this customer churn with the last contract expiration? (LEAVE, STAY, Unknown)
 - ID - Customer identifier

## **Part 1: Load Cleaned Data from Lab 6 and Preview it**

In this part of the lab, we will load the cleaned data from Lab 6. We had previously completed the following cleaning steps.

- Cleaned column names
- Fixed data types
- Handled missing values
- Removed duplicate records
- Reviewed for outliers

The link to the cleaned data is provided for your below. It contains 14,200 rows instead of 15,016 that we started out with in Lab 6.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/megatelco_cleaned.csv"
df = pd.read_csv(url)

df.sample(n =5)

In [ ]:
#check datatypes
df.info()

The only cleaning step we have to redo again is fixing data types. When we read in data, it once again infers data types.

In [ ]:
# Check original data types
print("Original dtypes:\n", df.dtypes)

# Convert object to nomimal categorical - can use df[colname].astype() to convert to nominal categorical
obj_to_nomcat_cols = ['considering_change_of_plan', 'college', 'operating_system', 'leave']
for acol in obj_to_nomcat_cols:
    df[acol] = df[acol].astype('category')

# Convert object/text columns with limited possible values with an order to ordinal categorical columnns
obj_to_ordcat_cols = ['reported_satisfaction', 'reported_usage_level']
for acol in obj_to_ordcat_cols:
    df[acol] = pd.Categorical(df[acol], categories = ['low', 'avg', 'high'], ordered = True)

# Check updated data types
print("\nUpdated dtypes:\n", df.dtypes)



In [ ]:
# View missing value counts
print("Missing values per column:\n", df.isnull().sum())

# Check for exact duplicates
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")


## **Part 2: Feature Engineering - Creating New Features**

**Data cleaning** makes the data more ***usable***.

After cleaning, the major next step is data transformation, getting data ready for predictive modeling and **feature engineering** — creating new columns from data to better capture useful patterns, and make it more ***useful***, meaningful and ready for Machine Learning.

In this and the next few sections, we will learn the following common techniques:

1. Feature engineering or creating new columns using builtin functions and algebra (+ - / *), that make patterns in the data more meaningful to ML models

2. Binning or discretizing continuous numeric variables using `pd.cut()` or `pd.qcut()` — to group them into bins (quantiles)

3. Scaling variables so that they are similar in range, or to reduce extreme values in data using log-scaling (`np.log()`), standardization (`StandardScaler`) and normalization (`MinMaxScaler`).

4. Encoding using either one-hot-encoding and `pd.get_dummies()` for nominal catgeorical variables (e.g., satisfaction levels) or integer encoding and `df.map()` for ordinal categorical variables

These new features help ML  models learn better, and thus makes these models more powerful (more predictive power), and may make results easier to interpret.

Things to think about:
- What are some new combinations (e.g., ratios, additions) of the existing columns that might be useful and meaningful to predict whether a customer would leave or stay?
- Can you create flag variables (a binary 0/1 variable) to highlight important traits (such as high data user, big texter etc) that you suspect might have interesting correlations with the target ?


In [ ]:
# Create a total data available variable (used + leftover)
df['total_data_mb'] = df['data_mb_used'] + df['data_leftover_mb']

# Create a ratio of overage to used data
df['overage_ratio'] = df['data_overage_mb'] / (df['data_mb_used'] + 1)  # add 1 to avoid divide-by-zero

# Create a binary flag for high texters (over 135 texts which is the mean)
df['high_texter'] = (df['text_message_count'] > 135).astype(int)

# Preview new columns
df[['total_data_mb', 'overage_ratio', 'high_texter']].head()


### 🔧 **Try It Yourself  Part 2**

2.1. Create a variable called `call_volume` by multiplying `over_15mins_calls_per_month` by `average_call_duration`

2.2. Create a binary flag `high_data_user` for users where `data_mb_used` is above the median

2.3. Use `.sample(n)` to check 10 of your new columns' values



In [ ]:
# 🔧 2.1. Add code here


# 🔧 2.2. Add code here


# 🔧 2.3. Add code here



## **Part 3: Binning Continous Variables**

Binning is the process of grouping numeric variables into categories (e.g., "low", "avg", "high").

### Why We Bin:
- Helps reduce the impact of outliers
- Allows us to use numeric values in models that prefer categories
- Simplifies interpretation and visualization

### Things to think about:
- Would binning values make patterns more visible?
- Do we want equal-sized groups or logical cutoffs?
- Is the variable skewed?

**Tools:**  
- `pd.qcut()` for quantile-based bins (equal frequency = same number of observations in each bin)  
- `pd.cut()` for equal-width bins (bins are of the same width) or custom size bins


In [ ]:
# Bin income into 3 equal obervation groups (quantiles): low, medium, high
df['income_group'] = pd.qcut(df['income'], q=3, labels=['low', 'medium', 'high'])

# Bin average call duration into 4 equal observation quartiles (labels = False will use integers 0,1,2,3 as group labels )
df['call_duration_group'] = pd.qcut(df['average_call_duration'], q=4, labels=False)

# Preview new groupings
df[['income', 'income_group', 'average_call_duration', 'call_duration_group']].sample(n=10)

### 🔧 **Try It Yourself - Part 3**

3.1. Use `pd.cut()` to group `data_mb_used` into 3 labeled bins: "light", "moderate", "heavy"

3.2. Use `pd.qcut()` on `text_message_count` to split into 4 equal-sized groups with integer labels

3.3. Print `.value_counts()` on each new column to see how values are distributed. Reminder: To ensure multiple outputs show, use `print()` or `display()`

In [ ]:
# 🔧 3.1. Add code here


# 🔧 3.2. Add code here


# 🔧 3.3. Add code here


## **Part 4: Scaling Numeric Variables**

Scaling transforms values to a common range (often 0–1), which helps many machine learning models perform better.

### When to Scale:
- When features have very different ranges (e.g., income vs. call duration)
- When using distance-based models (e.g., KNN, SVM) - they distance between observations to make decisions, and having variables of different units can distort results
- When comparing magnitudes across features (variables or columns)

### Common Methods:
- `Log scaling`: takes log of the column values; changes the shape of the distribution to make it *more normal*, reduces large extreme values, and expands small values
- `MinMaxScaler`: scales to 0-1 range; does NOT change the shape of the distribution, only rescales them, most sensitive to outlierd.
- `StandardScaler`: centers data around mean = 0 with unit variance (var = 1); also does NOT chnage the shape of the distribtion, only rescales them

### What scaling to use?
This is often not so clear-cut and we may try multiple techniques. Generally speaking,
- use log scaling if you want to change the shape to make a feature more normal. Distribution with right skew (large extreme values) often benefit from log scaling. Some (not all) algorithms will perform best when features are more normal. This also helps to remove/reduce far right outliers.
- Use minmax normalization when you want features to all be in the same ange 0-1 (but shapes remain whatever they are). This is most sensitive to outliers.
- Use standardization when your features are already somewhat normal, and you want them to have similar distribution (mean=0 and variance=1). Still sentsive to outliers, but not as much as minmax scaling.

### Things to think about:
- Are any features skewed to the right?
  - The histogram for handset_price shows that it's skewed to the right. This means there are a few very expensive handsets, but most are in a lower price range. Log scaling can help to reduce this skewness and make the distribution more symmetrical, which can improve the performance of some machine learning models that assume normally distributed data.
- Which features are on very different scales?
- Does the ML algorithm (that you plan to use in the Modeling phase) care about magnitude or distance?


**Tools:** `np.log()`, `MinMaxScaler`, `StandardScaler`

### Plot some histograms

Let us begin by plotting our original numeric variables to see their distributions. This will give us a clue as to what transformation to use.

- Anything that is skewed, let us log scale.
- Anything that looks close to normal, let us standardize.
- The rest we will minmax scale.

In [ ]:
cols_to_plot = ['income', 'data_overage_mb', 	'data_leftover_mb', 	'data_mb_used', 	'text_message_count', 	'house', 	'handset_price', 	'over_15mins_calls_per_month', 	'average_call_duration']

plt.figure(figsize=(10, 8)) # Figure height: (x, y)
for i, col in enumerate(cols_to_plot):
    plt.subplot(3, 3, i + 1) # plot a grid of 3 by 3 with 9 subplots
    df[col].hist()
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
# MinMaxSaler and StandardScaler are classes that need to be imported so we can use them
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# log scaling of right-skewed handset_price
df['price_log']= np.log(df['handset_price'])

# Initialize MinMaxScaler and StandardScaler
stdscaler = StandardScaler()
mmscaler = MinMaxScaler()

# below code will add one new column to df by standardizing values in one existing column
df['house_std'] = stdscaler.fit_transform(df[['house']])

# below code will normalize multiple columns at the same time into new columns in a new dataframe df_norm
cols_to_norm = ['income', 'data_mb_used']
df_norm = mmscaler.fit_transform(df[cols_to_norm])

# Add normalized columns back to our df
df['income_norm'] = df_norm[:, 0]
df['datambused_norm'] = df_norm[:, 1]

# Preview
df[['price_log','house_std', 'income_norm', 'datambused_norm', ]].head()

### **🔧 Try It Yourself - Part 4**

4.1. Scale `over_15mins_calls_per_month` using log scaling and add the new  column back to df with suffix "_log". Add a 1 to the values before taking log, because log 0 does not exist, or you will get -inf values.

4.2. Scale `average_call_duration` using StandardScaler and add the new  column back to df with suffix "_std"

4.3. Scale `text_message_count` using `MinMaxScaler` and add the new  column back to df with suffix "_norm".

4.4. Use `.describe()` to compare original vs. scaled versions and make a comment on what you observe


In [ ]:
# 🔧 4.1. Add code here


# 🔧 4.2. Add code here


# 🔧 4.3. Add code here


# 🔧 4.4. Add code here



🔧 4.4 Add comment here



## **Part 5: Encoding Categorical Variables**

Most machine learning models can't handle string categories directly (and can only use numeric values) —so we convert them into numbers using **encoding**.

### Types of Encoding:
- **One-hot encoding**: creates a binary column for each unique value in the original categorical column (**used for nominal variables**)
  - an example: We have a column called Colors with 3 values: red, blue, green - that we want to treat as nominal. OHE will create 3 new columns corresponidng to each color, and place a 1 in the new column, wherever the value in original column matches the color in its colname.
  - The original column is dropped by default, so it adds n new colums where n = number of unique values

|Color   |   red  | blue  | green  |
|--------|--------|-------|--------|
|red     |    1   |   0   |    0   |
|blue    |    0   |   1   |    0   |
|green   |    0   |   0   |    1   |
|red     |    1   |   0   |    0   |

- **Ordinal encoding**: assigns integers (**use only for ordered categories**)
  - An example: if we have a column called Education, we may want to set their values as 1,2,3,.....
  - adds only one new column, original column remains.

|Education   | educ_int  |
|--------|--------|
|some high school |     1     |
|high school grad |     2    |
|some college  |     3     |
|college grad |     4    |

### Things to consider:
- Is the variable nominal (e.g., OS type) or ordinal (e.g., satisfaction)?
- How many unique categories are there?
- Will one-hot encoding make the dataset too wide? Sometmes, if there are too many values, we may just keep it as string, buttne we lose the ability to use it in ML models. Nowadays with big data technologies, this is not an issue.

**Tool:** `pd.get_dummies()`, `df.map()`

In [ ]:
# Integer encode 'reported_usage_level' since it is nominal
df['usagelvl_encode'] = df['reported_usage_level'].map({'low': 1, 'avg': 2, 'high': 3})

# One-hot encode 'income_group' if we consider it nomimal - the column income_group will be dropped to avoid multicollinearity.
df = pd.get_dummies(df, columns=['income_group'], prefix='income')

# Preview new columns
#df.sample(n=10)
print(df.filter(like='usage').sample(n=10))
print(df.filter(like='income').sample(n=10))

### 🔧 **Try It Yourself - Part 5**

5.1. One-hot encode `operating_system` and integer encode `reported_satisfaction`.

5.2. Print `.shape` of your dataframe before and after to observe any big changes

5.3. How many new columns were added? explain. read the text in the previous Text cell to understand.


In [ ]:
# 🔧 5.1. Add code here


# 🔧 5.2. Add code here




🔧 5.3 Add comment here:

# 🔧 Part 6: Reflection (100 words or less per question)

6.1. Which transformation do you think had the biggest impact on preparing your data for modeling?

6.2. Are there any features you created that you think will be especially useful for predicting churn (leave (or stay))?

🔧 6.1. Add comment here:



🔧 6.2. Add comment here:

## Export Your Notebook to Submit in Canvas
- Use the instructions from Lab 1

In [ ]:
!jupyter nbconvert --to html "lab_07_LastnameFirstname.ipynb"